In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import PolynomialFeatures
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import SGDClassifier
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split


#випадкові ознаки
random_signs = 100

x1 = np.random.rand(random_signs)
x2 = np.random.rand(random_signs)

#матриця 2D 
x = np.column_stack((x1, x2))

def polynomial(x1, x2):
    y = 4*x1**2 + 5*x2**2 - 2*x1*x2 + 3*x1 - 6*x2
    return y

y_vector = polynomial(x1, x2)

print('Масив ознак:\n', x)
print("Форма X:\n", x.shape)
print("Масив змінної y:\n", y_vector)
print("Форма y:\n", y_vector.shape) 
print('----------')

#створення додаткових ознак для кожного степеня
def transform_polynom(x):
    polynom_object = PolynomialFeatures(degree=2, include_bias= False)
    x_polynom_object = polynom_object.fit_transform(x)
    return x_polynom_object

#cтандартизація даних
def data_scale(x):
    scaler = StandardScaler()
    return scaler.fit_transform(x)

data_scaled = data_scale(x)
df_scaled = pd.DataFrame(data_scaled, columns= ['Ознака1 ', 'Ознака 2'])
df_scaled['vector'] = y_vector
print('Виведення результату стандартизації:\n', df_scaled.describe())
print('----------')

x_b = np.hstack([np.ones((data_scaled.shape[0], 1)), df_scaled])

x_train, x_test, y_train, y_test = train_test_split(x_b, y_vector, test_size=0.2, random_state=100)

#Функції реалізації методів градієнтного спуску

def gradient_descent(x, y, lr=0.01, n_iter=1000):
    m, n = x.shape
    theta = np.zeros(n)
    for _ in range(n_iter):
        grad = x.T.dot(x.dot(theta) - y) / m
        theta -= lr * grad
    return theta

theta = gradient_descent(x_train, y_train)
print('Виведення коєфіцієнтів регресії методом градієнтного спуску:\n', theta)
print('----------')

#функції для різних методів градієнтного спуску

def stochastic_gradient_descent(X, y, lr=0.01, n_iter=2000):
    m, n = X.shape
    theta = np.zeros(n)
    for _ in range(n_iter):
        for i in range(m):
            idx = np.random.randint(m)
            xi = X[idx:idx+1]
            yi = y[idx]
            grad = xi.T.dot(xi.dot(theta) - yi)
            theta -= lr * grad
    return theta

def rmsprop(X, y, lr=0.01, n_iter=500, decay=0.9, eps=1e-8):
    m, n = X.shape
    theta = np.zeros(n)
    cache = np.zeros(n)
    for _ in range(n_iter):
        for i in range(m):
            xi = X[i:i+1]
            yi = y[i]
            grad = xi.T.dot(xi.dot(theta) - yi)
            cache = decay * cache + (1 - decay) * grad**2
            theta -= lr * grad / (np.sqrt(cache) + eps)
    return theta

def adam(X, y, lr=0.01, n_iter=100, beta1=0.9, beta2=0.999, eps=1e-8):
    m, n = X.shape
    theta = np.zeros(n)
    m_t = np.zeros(n)
    v_t = np.zeros(n)
    t = 0
    for _ in range(n_iter):
        for i in range(m):
            t += 1
            xi = X[i:i+1]
            yi = y[i]
            grad = xi.T.dot(xi.dot(theta) - yi)
            m_t = beta1 * m_t + (1 - beta1) * grad
            v_t = beta2 * v_t + (1 - beta2) * grad**2
            m_hat = m_t / (1 - beta1**t)
            v_hat = v_t / (1 - beta2**t)
            theta -= lr * m_hat / (np.sqrt(v_hat) + eps)
    return theta

def nadam(X, y, lr=0.01, n_iter=300, beta1=0.9, beta2=0.999, eps=1e-8):
    m, n = X.shape
    theta = np.zeros(n)
    m_t = np.zeros(n)
    v_t = np.zeros(n)
    t = 0
    for _ in range(n_iter):
        for i in range(m):
            t += 1
            xi = X[i:i+1]
            yi = y[i]
            grad = xi.T.dot(xi.dot(theta) - yi)
            m_t = beta1 * m_t + (1 - beta1) * grad
            v_t = beta2 * v_t + (1 - beta2) * grad**2
            m_hat = m_t / (1 - beta1**t)
            v_hat = v_t / (1 - beta2**t)
            theta -= lr * (beta1 * m_hat + (1 - beta1) * grad) / (np.sqrt(v_hat) + eps)
    return theta

#список функцій
optimizers = {"SGD": stochastic_gradient_descent, "RMSProp": rmsprop, "Adam": adam, "Nadam": nadam}

print(f"{'Алгоритм':<15} | {'Результат (theta)':<25}")
print("-" * 45)

for name, func in optimizers.items():
    theta_final = func(x, y_vector, lr=0.01, n_iter=50)
    theta_str = np.array2string(theta_final, precision=4)
    print(f"{name:<15} | {theta_str:<25}")
print('----------')


print("SGD:")
%timeit stochastic_gradient_descent(x, y_vector, n_iter=10)
print("\nRMSProp:")
%timeit rmsprop(x, y_vector, n_iter=10)
print("\nAdam:")
%timeit adamx(y_vector, n_iter=10)
print("\nNadam:")
%timeit nadam(x, y_vector, n_iter=10)





Масив ознак:
 [[0.31393114 0.49909885]
 [0.12630513 0.74935828]
 [0.80479074 0.85955963]
 [0.09787338 0.71625442]
 [0.10727849 0.44709501]
 [0.55734122 0.85410755]
 [0.65284512 0.58539226]
 [0.31532281 0.48536445]
 [0.33754554 0.75354956]
 [0.59826146 0.24670033]
 [0.64659816 0.25012751]
 [0.67778178 0.11754991]
 [0.9023663  0.38790728]
 [0.07671677 0.46629579]
 [0.54456384 0.07077368]
 [0.77509541 0.68863134]
 [0.47719828 0.22388612]
 [0.33601241 0.46189961]
 [0.12720701 0.67123992]
 [0.7835419  0.47345557]
 [0.43558061 0.56864788]
 [0.61601438 0.22423421]
 [0.36171736 0.80074826]
 [0.70012329 0.18695317]
 [0.50412505 0.67900323]
 [0.99423764 0.31294363]
 [0.26962397 0.14093474]
 [0.28312373 0.01109854]
 [0.00724182 0.35974931]
 [0.25760932 0.53788794]
 [0.14733485 0.81933665]
 [0.98494611 0.36123765]
 [0.66316459 0.81693795]
 [0.23645839 0.43739292]
 [0.00779482 0.44048859]
 [0.45662069 0.4489823 ]
 [0.77270228 0.12107883]
 [0.61707905 0.08548446]
 [0.11490907 0.03310611]
 [0.3516079

NameError: name 'X' is not defined